In [ ]:
import os
import subprocess
from IPython.display import Video
import cv2
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import subprocess

In [3]:
path = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition"


In [4]:
src = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/BalanceBeam/v_BalanceBeam_g01_c01.avi"
dst = "/kaggle/working/test.mp4"

subprocess.run(["ffmpeg", "-y", "-i", src, "-vcodec", "libx264", dst])

Video(dst, embed=True)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [5]:
!pip install ultralytics
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")
result = model.predict("/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/ApplyLipstick/v_ApplyLipstick_g01_c01.avi", save=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 3.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outpu

In [6]:
print(len(result))


for i , r in enumerate(result[:10]):
    # print(i , r.keypoints.xyn.shape)
    c = [r.keypoints.xyn.shape[0] for r in result]
    print(set(c))
    

149
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}
{1, 2}


In [ ]:
def pick_person(r):
    kp = r.keypoints.xyn.cpu().numpy()   # (تعداد آدم, 17, 2)
    if len(kp) == 0:
        return np.zeros((17, 2))         # هیچ‌کس نبود
    if len(kp) == 1:
        return kp[0]                     # یه نفر بود
    boxes = r.boxes.xywhn.cpu().numpy()  # اندازه‌ی جعبه‌ی هر آدم
    areas = boxes[:, 2] * boxes[:, 3]    # عرض × ارتفاع
    return kp[areas.argmax()]            # بزرگ‌ترین رو بردار

In [8]:
seq = np.array([pick_person(r) for r in result])
print(seq.shape)

(149, 17, 2)


In [9]:
def fix_length(seq, n=30):
    idx = np.linspace(0, len(seq) - 1, n).astype(int)
    return seq[idx]

x = fix_length(seq)
print(x.shape)

(30, 17, 2)


In [10]:
def normalize(x):
    # x: (30, 17, 2)
    hip = (x[:, 11] + x[:, 12]) / 2          # مرکز لگن
    shoulder = (x[:, 5] + x[:, 6]) / 2       # مرکز شانه‌ها
    scale = np.linalg.norm(shoulder - hip, axis=1)  # اندازه‌ی تنه
    scale[scale < 1e-6] = 1                  # جلوگیری از تقسیم بر صفر
    x = x - hip[:, None, :]                  # مرکز رو بیار روی صفر
    x = x / scale[:, None, None]             # اندازه رو یکسان کن
    return x

x_norm = normalize(x)
print(x_norm.shape)

(30, 17, 2)


In [11]:
def video_to_seq(path):
    results = model.predict(path, verbose=False)
    seq = np.array([pick_person(r) for r in results])
    seq = fix_length(seq)
    seq = normalize(seq)
    return seq.reshape(30, 34)

print(video_to_seq("/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train/ApplyLipstick/v_ApplyLipstick_g01_c01.avi").shape)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

(30, 34)


In [12]:
root = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/train"
print(sorted(os.listdir(root)))

['ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam', 'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress', 'Biking', 'Billiards', 'BlowDryHair', 'BlowingCandles', 'BodyWeightSquats', 'Bowling', 'BoxingPunchingBag', 'BoxingSpeedBag', 'BreastStroke', 'BrushingTeeth', 'CleanAndJerk', 'CliffDiving', 'CricketBowling', 'CricketShot', 'CuttingInKitchen', 'Diving', 'Drumming', 'Fencing', 'FieldHockeyPenalty', 'FloorGymnastics', 'FrisbeeCatch', 'FrontCrawl', 'GolfSwing', 'Haircut', 'HammerThrow', 'Hammering', 'HandstandPushups', 'HandstandWalking', 'HeadMassage', 'HighJump', 'HorseRace', 'HorseRiding', 'HulaHoop', 'IceDancing', 'JavelinThrow', 'JugglingBalls', 'JumpRope', 'JumpingJack', 'Kayaking', 'Knitting', 'LongJump', 'Lunges', 'MilitaryParade', 'Mixing', 'MoppingFloor', 'Nunchucks', 'ParallelBars', 'PizzaTossing', 'PlayingCello', 'PlayingDaf', 'PlayingDhol', 'PlayingFlute', 'PlayingGuitar', 'PlayingPiano', 'PlayingSitar', 'PlayingTabla', 'P

In [13]:
classes = [
    "JumpRope",          # طناب‌زدن
    "JumpingJack",       # پرش ستاره‌ای
    "PushUps",           # شنا
    "PullUps",           # بارفیکس
    "BodyWeightSquats",  # اسکوات
    "Lunges",            # لانج
    "Punch",             # مشت‌زنی
    "TaiChi",            # تای‌چی (حرکت آهسته)
    "HandstandPushups",  # شنای دست‌به‌زمین
    "SumoWrestling",     # کشتی سومو
]

In [ ]:
X, y = [], []

for label, cls in enumerate(classes):
    folder = os.path.join(root, cls)
    for name in tqdm(os.listdir(folder), desc=cls):
        try:
            X.append(video_to_seq(os.path.join(folder, name)))
            y.append(label)
        except Exception as e:
            print("خطا:", name, e)

X = np.array(X)
y = np.array(y)
print(X.shape, y.shape)

np.save("/kaggle/working/X_train.npy", X)
np.save("/kaggle/working/y_train.npy", y)

JumpRope:   0%|          | 0/108 [00:00<?, ?it/s]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



JumpRope:   1%|          | 1/108 [00:04<08:36,  4.83s/it]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



JumpRope:   2%|▏         | 2/108 [00:09<08:26,  4.78s/it]

...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



SumoWrestling:  99%|█████████▉| 86/87 [03:38<00:02,  2.26s/it]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



SumoWrestling: 100%|██████████| 87/87 [03:42<00:00,  2.56s/it]

(908, 30, 34) (908,)


In [ ]:
X = np.load("/kaggle/working/X_train.npy")
y = np.load("/kaggle/working/y_train.npy")

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_tr.shape, X_val.shape)

(726, 30, 34) (182, 30, 34)


In [ ]:
model_lstm = keras.Sequential([
    layers.Input(shape=(30, 34)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation="softmax"),
])

model_lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model_lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 30, 128)        │        83,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 133,514 (521.54 KB)

 Trainable params: 133,514 (521.54 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model_lstm.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=16,
)

Epoch 1/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9711 - loss: 0.0995 - val_accuracy: 0.8626 - val_loss: 0.6265
Epoch 2/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9628 - loss: 0.1422 - val_accuracy: 0.8132 - val_loss: 0.7171
Epoch 3/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9738 - loss: 0.1100 - val_accuracy: 0.8352 - val_loss: 0.6682
Epoch 4/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9573 - loss: 0.1587 - val_accuracy: 0.8736 - val_loss: 0.5703
Epoch 5/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9904 - loss: 0.0330 - val_accuracy: 0.8736 - val_loss: 0.6040
Epoch 6/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9504 - loss: 0.1638 - val_accuracy: 0.8132 - val_loss: 0.6525
Epoch 7/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9752 - loss: 0.0876 - val_accuracy: 0.8736 - val_loss: 0.5162
Epoch 8/40
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9959 - loss: 0.0257 - val_accuracy: 0.8736 - v

In [24]:
root_test = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test"

X_test, y_test = [], []

for label, cls in enumerate(classes):
    folder = os.path.join(root_test, cls)
    for name in tqdm(os.listdir(folder), desc=cls):
        try:
            X_test.append(video_to_seq(os.path.join(folder, name)))
            y_test.append(label)
        except Exception as e:
            print("خطا:", name, e)

X_test = np.array(X_test)
y_test = np.array(y_test)

loss, acc = model_lstm.evaluate(X_test, y_test)
print("دقت روی test:", acc)

JumpRope:   0%|          | 0/18 [00:00<?, ?it/s]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



JumpRope:   6%|▌         | 1/18 [00:05<01:26,  5.08s/it]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



JumpRope:  11%|█         | 2/18 [00:09<01:16,  4.81s/it]

...

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



SumoWrestling:  93%|█████████▎| 14/15 [00:34<00:02,  2.03s/it]

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



SumoWrestling: 100%|██████████| 15/15 [00:35<00:00,  2.39s/it]

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8961 - loss: 0.5445
دقت روی test: 0.8961039185523987


In [26]:
model_lstm.save("/kaggle/working/pose_lstm.keras")

In [31]:
def predict_video(path):
    seq = video_to_seq(path)                    # (30, 34)
    probs = model_lstm.predict(seq[None], verbose=0)[0]
    best = probs.argmax()
    return classes[best], probs[best]

name, conf = predict_video("/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/BabyCrawling/v_BabyCrawling_g10_c04.avi")
print(name, conf)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

SumoWrestling 0.9994711


In [32]:
def draw_video(path, out_path="/kaggle/working/result.mp4"):
    name, conf = predict_video(path)
    results = model.predict(path, verbose=False)

    h, w = results[0].orig_shape
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))

    for r in results:
        frame = r.plot()   # فریم با اسکلت کشیده‌شده
        cv2.putText(frame, f"{name} {conf:.0%}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 255, 0), 1)
        writer.write(frame)

    writer.release()
    return out_path

out = draw_video("/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/BalanceBeam/v_BalanceBeam_g10_c02.avi")

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks out

In [ ]:
subprocess.run(["ffmpeg", "-y", "-i", out, "-vcodec", "libx264", "/kaggle/working/final.mp4"])
Video("/kaggle/working/final.mp4", embed=True)

In [27]:
classes

['JumpRope',
 'JumpingJack',
 'PushUps',
 'PullUps',
 'BodyWeightSquats',
 'Lunges',
 'Punch',
 'TaiChi',
 'HandstandPushups',
 'SumoWrestling']

In [ ]:
def draw_label(frame, text , true_name):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), base = cv2.getTextSize(text, font, 0.9, 2)
    cv2.rectangle(frame, (0, 0), (520 , 18 ),
                  (255, 255, 255), -1)
    text = text.split(" ")
    cv2.putText(frame, f"pre: {text[0]} | coef: {text[1]}", (10, 12), font, 0.45, (255, 0, 0), 1 , cv2.LINE_AA)
    # cv2.putText(frame, f"true: {true_name}", (10, 100), font, 0.45, (255, 255, 0), 1 , cv2.LINE_AA)
    

    return frame


def draw_video(path, true_name ,  out_path="/kaggle/working/result.mp4"):
    name, conf = predict_video(path)               # پیش‌بینی کل ویدیو
    results = model.predict(path, verbose=False)   # اسکلت هر فریم

    h, w = results[0].orig_shape
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))

    for r in results:
        frame = r.plot(font_size=8, line_width=1)                               # فریم + اسکلت
        frame = draw_label(frame, f"{name} {conf:.0%}" , true_name)  # باکس اسم کلاس
        writer.write(frame)

    writer.release()
    return out_path
 

class_name = classes[7]
p = f"/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/{class_name}/v_{class_name}_g01_c03.avi"
p = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/JumpRope/v_JumpRope_g17_c04.avi"

print(p)
out = draw_video(p, None)

# تبدیل به فرمتی که نوت‌بوک پخش کنه
subprocess.run(["ffmpeg", "-y", "-i", out, "-vcodec", "libx264", "/kaggle/working/final.mp4"])
Video("/kaggle/working/final.mp4", embed=True)

/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/JumpRope/v_JumpRope_g17_c04.avi
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab